In [177]:
import pandas as pd
import numpy as np

In [178]:
postings = pd.read_csv("/Users/aishanipradhan/Desktop/data_visualization/eda/archive/postings.csv")
postings.shape

(123849, 31)

In [179]:
postings = postings[~postings["company_name"].isna()]
postings.shape

(122130, 31)

## Joining finance dataset

In [180]:
finance_mapping = pd.read_csv("/Users/aishanipradhan/Desktop/prof_skills/project/data_creation/finance_mapping.csv")
master_df = postings.merge(finance_mapping, how='left', on='company_name')
master_df.shape

(122130, 32)

In [181]:
master_df = master_df[~master_df["finance_mapping"].isna()]
master_df.shape

(2477, 32)

In [182]:
finance = pd.read_csv("/Users/aishanipradhan/Desktop/prof_skills/project/data_creation/final_financial_dataframe.csv")

In [183]:
master_df = master_df.merge(finance, how='left', left_on='finance_mapping', right_on='Company')
master_df.shape

(2488, 54)

## Join Follower Count

In [184]:
employee_counts = pd.read_csv("/Users/aishanipradhan/Desktop/data_visualization/eda/archive/companies/employee_counts.csv")

In [185]:
# Take latest recorded folower count
employee_counts_max = employee_counts.loc[
    employee_counts.groupby('company_id')['time_recorded'].idxmax()
]

In [186]:
employee_counts_max = employee_counts_max[["company_id","follower_count"]]
master_df = master_df.merge(employee_counts_max, on="company_id", how="left")
master_df.shape

(2488, 55)

## Cleaning Master

In [187]:
master_df = master_df.drop(columns=[
    'description',
    'max_salary',
    'pay_period',
    'med_salary',
    'min_salary',
    'formatted_work_type',
    'applies',
    'original_listed_time',
    'remote_allowed',
    'job_posting_url',
    'application_url',
    'application_type',
    'expiry',
    'closed_time',
    'listed_time',
    'posting_domain',
    'sponsored',
    'work_type',
    'currency',
    'compensation_type',
    'normalized_salary',
    'zip_code',
    'fips',
    'Industry.1'
])

In [188]:
master_df = master_df.rename(columns={'company_name': 'company_linkedin'})
master_df.columns = [col.lower() for col in master_df.columns]

In [189]:
# --- String columns to clean ---
string_cols = ['company_linkedin', 'company', 'title', 'location', 'formatted_experience_level', 'skills_desc']

for col in string_cols:
    master_df[col] = (
        master_df[col]
        .astype(str)
        .str.strip()                     # remove leading/trailing spaces
        .str.replace(r'\s+', ' ', regex=True)  # normalize internal whitespace
        .str.lower()                     # make lowercase
        .replace('nan', np.nan)          # convert literal "nan" strings back to NaN
    )


/var/folders/_5/zwklnk315yqbrfhydp3_pj5w0000gn/T/ipykernel_3613/1869818270.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace('nan', np.nan)          # convert literal "nan" strings back to NaN


In [190]:
#master_df.to_csv("/Users/aishanipradhan/Desktop/master_df_without_skills.csv")

## (Optional) Joining Skills

In [191]:
skills_abr = pd.read_csv("/Users/aishanipradhan/Desktop/data_visualization/eda/archive/jobs/job_skills.csv")
skills_abr.shape

(213768, 2)

In [192]:
skills_full = pd.read_csv("/Users/aishanipradhan/Desktop/data_visualization/eda/archive/mappings/skills.csv")

In [193]:
skills = skills_abr.merge(skills_full, on = "skill_abr")

In [194]:
skills = (
    skills.groupby('job_id', as_index=False)
    .agg({
        'skill_abr': lambda x: ', '.join(sorted(set(x))),
        'skill_name': lambda x: ', '.join(sorted(set(x)))
    })
)

In [195]:
skills.head()

,job_id,skill_abr,skill_name
0,921716,"MRKT, SALE","Marketing, Sales"
1,1218575,HCPR,Health Care Provider
2,1829192,HCPR,Health Care Provider
3,2264355,"ART, DSGN, IT","Art/Creative, Design, Information Technology"
4,10998357,"MGMT, MNFC","Management, Manufacturing"


In [196]:
master_df.shape

(2488, 31)

In [197]:
master_df = master_df.merge(skills, on = "job_id")
master_df.shape

(2488, 33)

In [198]:
master_df.to_csv("/Users/aishanipradhan/Desktop/master_df_with_skills.csv")